In [1]:
#Import necessary libraries (e.g., pandas, numpy)
import pandas as pd
import re
import nltk
import spacy

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package

In [2]:
# load data
dataset = "IMDB Dataset.csv" # read  file
# Read the CSV file into a dataframe.
df = pd.read_csv(dataset)

In [3]:
# Number of samples
print("Number of samples:", df.shape[0])

# Columns
print("Columns:", df.columns)

df.head()

Number of samples: 50000
Columns: Index(['review', 'sentiment'], dtype='str')


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Step 2: Data Exploration (EDA) 
# Check for: Missing values & Duplicate entries
# Print missing values
print(df.isnull().sum())
# Count duplicate rows
print("Duplicate rows: ", df.duplicated().sum())
# drops duplicates
df = df.drop_duplicates()

review       0
sentiment    0
dtype: int64
Duplicate rows:  418


In [5]:
# Analyze:
# Class distribution (positive vs negative)
print(df['sentiment'].value_counts())
# Length of reviews (short vs long)
df['review_length'] = df['review'].apply(len)
df['review_length'].describe()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64


count    49582.000000
mean      1310.568230
std        990.762238
min         32.000000
25%        699.000000
50%        971.000000
75%       1592.000000
max      13704.000000
Name: review_length, dtype: float64

In [6]:
# Inspect:
#Raw reviews (look for noise such as HTML tags, punctuation)
print(df['review'].iloc[0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [7]:
#Step 3: Train-Test Split (80 - 20) (5 marks)
#Split your dataset before any step that learns from data.
#Any operation using fit() must be applied only on training data.
#The same transformation must be applied to test data using transform().
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 39665
Test size: 9917


In [ ]:
# Step 4: Text Preprocessing (8 marks)
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove: HTML tags punctuation special characters
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # 3. Tokenize text into words
    words = text.split()
    # 4. Remove stopwords
    words = [w for w in words if w not in stop_words]
    # 5. Apply: Stemming Lemmatization
    words = [stemmer.stem(w) for w in words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

X_train_clean = X_train.apply(preprocess)
X_test_clean = X_test.apply(preprocess)

In [9]:
# test print to see if works
print("Before:", X_train.iloc[0])
print("After:", X_train_clean.iloc[0])

Before: I really liked the movie 'The Emporer's New Groove', but watching this was like coming home and seeing your wife having "relations" with a llama. Seriously, this movie was bad. It's like Club Dread after Super Troopers. I am supposed to write 10 lines, but I don't even know what else to say. I laughed a couple of times, but only because I was drinking. A movie like that should at least be funny when your drunk. It was not. Maybe llamas are just funny and regular cartoon people aren't. Either way, just stick with The Emporer's New Groove if you want a funny, cartoon, llama-themed movie. Line 10 is this line right here.
After: i realli like movi the empor new groov watch like come home see wife relat llama serious movi bad it like club dread super trooper i suppos write line i even know el say i laugh coupl time i drink a movi like least funni drunk it mayb llama funni regular cartoon peopl either way stick the empor new groov want funni cartoon llama theme movi line line right


In [10]:
# Step 5: Linguistic Analysis (6 marks)
#Perform POS Tagging
nlp = spacy.load("en_core_web_sm")
tokens = nltk.word_tokenize(X_train_clean.iloc[0])
pos_tags = nltk.pos_tag(tokens)
print(pos_tags)
#Perform Named Entity Recognition (NER)
doc = nlp(X_train.iloc[0])
for ent in doc.ents:
    print(ent.text, ent.label_)

# Answer the following:
#Which POS tags are most relevant for sentiment?
# good, bad, very, extremely
#Do entities (e.g., actors, movies) influence sentiment?
# Yes! The actor's names are tied to the positive or negative reviews

[('i', 'JJ'), ('realli', 'VBP'), ('like', 'IN'), ('movi', 'PDT'), ('the', 'DT'), ('empor', 'JJ'), ('new', 'JJ'), ('groov', 'NN'), ('watch', 'NN'), ('like', 'IN'), ('come', 'JJ'), ('home', 'NN'), ('see', 'VBP'), ('wife', 'NN'), ('relat', 'JJ'), ('llama', 'RB'), ('serious', 'JJ'), ('movi', 'NN'), ('bad', 'JJ'), ('it', 'PRP'), ('like', 'IN'), ('club', 'NN'), ('dread', 'VBP'), ('super', 'IN'), ('trooper', 'NN'), ('i', 'NN'), ('suppos', 'VBP'), ('write', 'JJ'), ('line', 'NN'), ('i', 'NN'), ('even', 'RB'), ('know', 'VBP'), ('el', 'NNS'), ('say', 'VBP'), ('i', 'JJ'), ('laugh', 'IN'), ('coupl', 'JJ'), ('time', 'NN'), ('i', 'JJ'), ('drink', 'VBP'), ('a', 'DT'), ('movi', 'NN'), ('like', 'IN'), ('least', 'JJS'), ('funni', 'JJ'), ('drunk', 'NN'), ('it', 'PRP'), ('mayb', 'VBZ'), ('llama', 'JJ'), ('funni', 'NN'), ('regular', 'JJ'), ('cartoon', 'NN'), ('peopl', 'NN'), ('either', 'DT'), ('way', 'NN'), ('stick', 'VBD'), ('the', 'DT'), ('empor', 'JJ'), ('new', 'JJ'), ('groov', 'NN'), ('want', 'VBP'), ('

In [11]:
# Step 6: Text Representation (7 marks)
# You must implement two types of representations:
# 6A: TF-IDF (Baseline)
# Convert text into numerical vectors using TF-IDF.
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)
# test
print(X_train_tfidf.shape)

(39665, 64065)


In [12]:
# 6B: Document Embeddings (Semantic) Use Doc2Vec
# Tag documents
train_tagged = [
   TaggedDocument(words=doc.split(), tags=[i])
   for i, doc in enumerate(X_train_clean)
]
# Train model
doc2vec_model = Doc2Vec(vector_size=100, window=5, min_count=2, epochs=20)
doc2vec_model.build_vocab(train_tagged)
doc2vec_model.train(train_tagged, total_examples=doc2vec_model.corpus_count, epochs=20)
# Create vectors
X_train_d2v = [doc2vec_model.infer_vector(doc.split()) for doc in X_train_clean]
X_test_d2v = [doc2vec_model.infer_vector(doc.split()) for doc in X_test_clean]
#test
print(len(X_train_d2v), len(X_train_d2v[0]))

39665 100


In [13]:
# Step 7: Model Development  (4 marks)
# You must train two models & Train each model on: TF-IDF features & Embedding-based features

# Logistic Regression 
lr = LogisticRegression()
# TF-IDF
lr.fit(X_train_tfidf, y_train)
y_pred_lr_tfidf = lr.predict(X_test_tfidf)
# Doc2Vec
lr.fit(X_train_d2v, y_train)
y_pred_lr_d2v = lr.predict(X_test_d2v)

# Random Forest
rf = RandomForestClassifier()
# TF-IDF
rf.fit(X_train_tfidf, y_train)
y_pred_rf_tfidf = rf.predict(X_test_tfidf)
# Doc2Vec
rf.fit(X_train_d2v, y_train)
y_pred_rf_d2v = rf.predict(X_test_d2v)

In [14]:
# Step 8: Model Evaluation ( 4 marks)
# Evaluate models using:
def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    # Accuracy
    print("Accuracy:", accuracy_score(y_test, y_pred))
    # Precision
    print("Precision:", precision_score(y_test, y_pred, pos_label='positive'))
    # Recall
    print("Recall:", recall_score(y_test, y_pred, pos_label='positive'))
    # F1-score
    print("F1-score:", f1_score(y_test, y_pred, pos_label='positive'))
    
evaluate(y_test, y_pred_lr_tfidf, "LR + TFIDF")
evaluate(y_test, y_pred_lr_d2v, "LR + Doc2Vec")
evaluate(y_test, y_pred_rf_tfidf, "RF + TFIDF")
evaluate(y_test, y_pred_rf_d2v, "RF + Doc2Vec")


LR + TFIDF
Accuracy: 0.8896843803569628
Precision: 0.8772338772338772
Recall: 0.907191643230213
F1-score: 0.891961287774047

LR + Doc2Vec
Accuracy: 0.8505596450539478
Precision: 0.855573637103336
Recall: 0.844917637605464
F1-score: 0.8502122498483929

RF + TFIDF
Accuracy: 0.8460219824543713
Precision: 0.8464163822525598
Recall: 0.8469264764965849
F1-score: 0.8466713525454362

RF + Doc2Vec
Accuracy: 0.8120399314308763
Precision: 0.8126506024096386
Recall: 0.8129770992366412
F1-score: 0.8128138180357501
